<a href="https://colab.research.google.com/github/DOYEON-DOT/ImagineerProject/blob/main/HumeAI_API_Test%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hume AI API Test



>기쁨(Happiness) 음성 샘플 3개를 Hume AI EVI 3(Prosody) 모델로 분석하고, 가장 높은 점수의 세부 감정을 추출하여 결과를 출력하는 코드


In [ ]:
!pip install -q hume pandas

import time
import shutil
import pandas as pd
from google.colab import files

from hume.client import HumeClient
from hume.expression_measurement.batch.types import (
    InferenceBaseRequest,
    Models,
    Prosody
)

# ==========================================
# API KEY
# ==========================================
HUME_API_KEY = ""

client = HumeClient(api_key=HUME_API_KEY)

# ==========================================
# 기쁨 음성 3개 업로드
# ==========================================
file_paths = []

for i in range(3):
    print(f"happiness 음성 {i+1}번째 파일 업로드")

    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]
    file_paths.append(file_path)

print("\n원본 파일명")
print(file_paths)

# ==========================================
# Hume 전송용 파일명 변경
# ==========================================
safe_file_paths = []

for i, path in enumerate(file_paths, start=1):
    new_name = f"happiness_{i}.wav"
    shutil.copy(path, new_name)
    safe_file_paths.append(new_name)

print("\nHume 전송 파일명")
print(safe_file_paths)

# ==========================================
# Hume v3 Prosody 설정
# ==========================================
config = InferenceBaseRequest(
    models=Models(
        prosody=Prosody()
    )
)

# ==========================================
# 분석 요청
# ==========================================
audio_files = [open(path, "rb") for path in safe_file_paths]

job = client.expression_measurement.batch.start_inference_job_from_local_file(
    json=config,
    file=audio_files
)

for f in audio_files:
    f.close()

# ==========================================
# Job ID
# ==========================================
if isinstance(job, str):
    job_id = job
elif hasattr(job, "job_id"):
    job_id = job.job_id
elif hasattr(job, "id"):
    job_id = job.id
else:
    job_id = str(job)

print("\nJob ID:", job_id)

# ==========================================
# 완료 대기
# ==========================================
while True:
    details = client.expression_measurement.batch.get_job_details(
        id=job_id
    )

    status = details.state.status
    print("현재 상태:", status)

    if status == "COMPLETED":
        break

    if status == "FAILED":
        raise Exception("분석 실패")

    time.sleep(5)

# ==========================================
# 결과 가져오기
# ==========================================
results = client.expression_measurement.batch.get_job_predictions(
    id=job_id
)

# ==========================================
# 결과 정리
# ==========================================
rows = []

for idx, result in enumerate(results):
    original_file = file_paths[idx]

    if not result.results.predictions:
        rows.append({
            "원본 파일명": original_file,
            "실제 라벨": "happiness",
            "Hume 세부감정": "분석 결과 없음",
            "점수": ""
        })
        continue

    prediction = result.results.predictions[0]

    emotions = (
        prediction.models.prosody.grouped_predictions[0]
        .predictions[0]
        .emotions
    )

    top_emotion = max(
        emotions,
        key=lambda x: x.score
    )

    rows.append({
        "원본 파일명": original_file,
        "실제 라벨": "happiness",
        "Hume 세부감정": top_emotion.name,
        "점수": round(top_emotion.score, 4)
    })

df = pd.DataFrame(rows)

print("\n===== happiness 음성 3개 분석 결과 =====")

display(df)